### Get data from Database

In production the query must be filtered by query parameters passed in by:
- System Rules:
    - Date Posted (Previous 30 days?)
- AI results:
    - Industries
    - Title
- Selected by the user:
    - Country
    - Location

In [13]:
import pandas as pd
import psycopg2
from os import getenv

In [14]:
database_user = getenv("DB_USER")
database_password = getenv("DB_PASSWORD")
HOST = "localhost"
DATABASE = "market_fit"
TABLE_NAME = "job_postings"

with psycopg2.connect(
    host=HOST,
    database=DATABASE,
    user=database_user,
    password=database_password
) as conn:

    if not conn:
        print("Connection to the database failed!")
        exit(1)

    cur = conn.cursor()
    cur.execute(f"SELECT * FROM {TABLE_NAME}")  
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=[desc[0] for desc in cur.description])
    df = df[["id", "title", "description"]]

### Extract Skills From Positions

In [15]:
import pandas as pd
from typing import List, Optional
from google import genai
from pydantic import BaseModel, Field
from multiprocessing.pool import ThreadPool
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model 

DEFAULT_REASONING_LLM_MODEL = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.5
api_key = getenv('GEMINI_API_KEY')

# ── Models ────────────────────────────────────────────────────────────────────

class HardSkill(BaseModel):
    description: str = Field(description="Objective description of the hard skill or experience in English.")
    time_experience: Optional[float] = Field(description="Optional time experience in months identified for the hard skill application.")
    weight: Optional[float] = Field(description="Optional weighted relevance score for the hard skill application in general for the position, ranging from 0 to 1.")

class HardSkillList(BaseModel):
    hard_skills: List[HardSkill] = Field(description="List of hard skills extracted from a job description.")

class SoftSkill(BaseModel):
    description: str = Field(description="Concise canonical name of the soft skill in English (e.g. 'Stakeholder Communication', 'Proactive Communication').")
    weight: Optional[float] = Field(description="Weighted relevance score for this soft skill for the position, ranging from 0 to 1.")

class SoftSkillList(BaseModel):
    soft_skills: List[SoftSkill] = Field(description="List of soft skills extracted from a job description.")

# ── Extractors ────────────────────────────────────────────────────────────────

def _build_llm(model_name=DEFAULT_REASONING_LLM_MODEL) -> genai.Client:
    return init_chat_model(
        model=model_name,
        model_provider=LLM_PROVIDER,
        temperature=LLM_TEMPERATURE,
        api_key=api_key
    )

def embed_skill(chat_model: genai.Client, skill: str) -> List[float]:
    try:
        response = chat_model.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=[skill]
        )
    except Exception as e:
        print(f"Error embedding skill '{skill}': {e}")
        return []
    return response.embeddings[0].values

def extract_hard_skills_list(chat_model: genai.Client, text: str) -> List[HardSkill]:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            You are an expert technical recruiter and skills taxonomy specialist.
            Your task is to extract ALL hard skills from a job description and assign each a relevance weight.

            ## Extraction Rules

            1. **Granularity**: Extract BOTH the broad category AND each specific tool/technology mentioned within it.
            - Example: "ETL (SSIS or Azure)" → extract `ETL`, `SSIS`, and `Azure` as separate skills.

            2. **Descriptions**: Use short, normalized names in English (e.g. `SQL`, `Python`, `ETL`, `Azure Data Factory`).
            - Avoid long sentences — prefer concise canonical names.
            - If a skill implies a category (e.g. "Programming language (Python preferred)"), extract both `Programming` and `Python`.
            - For architecture/design skills, prefer the canonical compound form: e.g. "data pipeline and analytics architectures" → extract `Data Pipeline Architecture` and `Data Architecture` as separate entries, NOT just `Data Pipelines` or `Data Analysis`.

            3. **Weight assignment** — assign a `weight` from 0.0 to 1.0 based on how the job description frames the skill:
            | Signal in job description              | Weight range |
            |----------------------------------------|--------------|
            | Required / mandatory / solid knowledge | 0.9 – 1.0    |
            | Preferred / desirable, named directly  | 0.6 – 0.8    |
            | Desirable but more generic/optional    | 0.4 – 0.6    |
            | Nice-to-have / loosely implied         | 0.1 – 0.3    |

            - Parent categories of required tools should inherit a high weight (e.g. `ETL` is mandatory, so weight = 1.0).
            - Specific tools listed under "preferred" should have slightly lower weight than their category.

            4. **Language**: All descriptions must be in English, even if the job description is in another language.
            5. **Language proficiency skills**: Always extract language requirements (e.g. "fluent English", "written and spoken English") as hard skills. Use the format `Fluent English` or `English Proficiency`. Assign weight based on how it is framed (required vs. desirable).
            6. **time_experience**: Only populate this field if the job description explicitly mentions a duration (e.g. "3+ years of Python"). Otherwise leave it null.
        """),
        ("human", "{input}")
    ])

    structured_llm = chat_model.with_structured_output(schema=HardSkillList)
    chain = prompt | structured_llm
    response = chain.invoke({"input": text})
    return response.hard_skills

def extract_soft_skills_list(chat_model: genai.Client, text: str) -> List[SoftSkill]:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """
            You are an expert recruiter and organizational psychologist specializing in behavioral competency frameworks.
            Your task is to extract ALL soft skills and behavioral competencies from a job description and assign each a relevance weight.

            ## What counts as a soft skill
            Soft skills are interpersonal, behavioral, or cognitive traits that are NOT tied to a specific tool, technology, or domain.
            Examples: `Stakeholder Communication`, `Cross-functional Collaboration`, `Proactive Communication`, `Adaptability`, `Problem Solving`.

            ## What does NOT count as a soft skill
            - Tool or technology proficiency (e.g. SQL, Python, Power BI) → these are hard skills, ignore them
            - Language proficiency (e.g. Fluent English) → treated as a hard skill, ignore it here
            - Domain knowledge (e.g. Data Modeling, ETL) → hard skill, ignore it

            ## Extraction Rules

            1. **Descriptions**: Use short canonical names in English (e.g. `Stakeholder Communication`, `Proactive Risk Escalation`).
            - Avoid full sentences. Prefer noun phrases.
            - If a sentence implies multiple distinct soft skills, extract each separately.

            2. **Weight assignment** — assign a `weight` from 0.0 to 1.0:
            | Signal in job description                        | Weight range |
            |--------------------------------------------------|--------------|
            | Strongly emphasized / required behavioral trait  | 0.8 – 1.0    |
            | Clearly mentioned but not the main focus         | 0.5 – 0.7    |
            | Implied or nice-to-have                          | 0.2 – 0.4    |

            3. **Language**: All descriptions must be in English, even if the job description is in another language.
        """),
        ("human", "{input}")
    ])

    structured_llm = chat_model.with_structured_output(schema=SoftSkillList)
    chain = prompt | structured_llm
    response = chain.invoke({"input": text})
    return response.soft_skills


# ── Row processors ────────────────────────────────────────────────────────────

def process_row_hard_skills(chat_model: genai.GenerativeModel, row: dict) -> tuple:
    job_id = row['id']
    text = f"Job title: {row['title']}\nJob description: {row['description']}"
    try:
        skills = extract_hard_skills_list(chat_model, text)
    except Exception as e:
        print(f"[hard] Error processing job_id {job_id}: {e}")
        skills = []
    return job_id, skills

def process_row_soft_skills(chat_model: genai.GenerativeModel, row: dict) -> tuple:
    job_id = row['id']
    text = f"Job title: {row['title']}\nJob description: {row['description']}"
    try:
        skills = extract_soft_skills_list(chat_model, text)
    except Exception as e:
        print(f"[soft] Error processing job_id {job_id}: {e}")
        skills = []
    return job_id, skills


# ── DataFrame builders ────────────────────────────────────────────────────────

def build_hard_skills_df(chat_model: genai.GenerativeModel, df: pd.DataFrame) -> pd.DataFrame:
    rows = df.to_dict(orient='records')
    with ThreadPool() as pool:
        results = pool.map(lambda row: process_row_hard_skills(chat_model, row), rows)

    records = []
    for job_id, skills in results:
        for skill in skills:
            records.append({
                "job_id": job_id,
                "skill_description": skill.description,
                "time_experience": skill.time_experience,
                "weight": skill.weight
            })
    return pd.DataFrame(records)

def build_soft_skills_df(chat_model: genai.GenerativeModel, df: pd.DataFrame) -> pd.DataFrame:
    rows = df.to_dict(orient='records')
    with ThreadPool() as pool:
        results = pool.map(lambda row: process_row_soft_skills(chat_model, row), rows)

    records = []
    for job_id, skills in results:
        for skill in skills:
            records.append({
                "job_id": job_id,
                "skill_description": skill.description,
                "weight": skill.weight
            })
    return pd.DataFrame(records)


### Create Embeddings

In [16]:
import time
import re
from google.genai import errors

client = genai.Client(api_key=api_key)

def embed_batch_with_retry(skills: List[str], max_retries: int = 3) -> List[List[float]]:
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=skills
            )
            return [e.values for e in response.embeddings]
        except errors.ClientError as e:
            is_rate_limit = (e.code == 429 or "RESOURCE_EXHAUSTED" in str(e.status))
            if not is_rate_limit or attempt == max_retries - 1:
                raise

            match = re.search(r'retry in (\d+(?:\.\d+)?)s', str(e))
            wait = float(match.group(1)) if match else (2 ** attempt * 10)
            print(f"Rate limited. Waiting {wait:.0f}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait)

    raise RuntimeError("Max retries exceeded")

def build_embeddings(df: pd.DataFrame, batch_size: int = 100, concurrency: int = 2) -> pd.DataFrame:
    result_df = df.copy()
    skills = result_df["skill_description"].tolist()

    batches = [skills[i:i + batch_size] for i in range(0, len(skills), batch_size)]

    with ThreadPool(concurrency) as pool:
        batch_results = pool.map(embed_batch_with_retry, batches)

    embeddings = [emb for batch in batch_results for emb in batch]
    result_df["embedding"] = embeddings
    return result_df

In [ ]:
chat_model = _build_llm()

hard_skills_df = build_hard_skills_df(chat_model, df)
soft_skills_df = build_soft_skills_df(chat_model, df)
hard_skills_df_with_embeddings = build_embeddings(hard_skills_df)
soft_skills_df_with_embeddings = build_embeddings(soft_skills_df)

Rate limited. Waiting 21s before retry 1/3...
Rate limited. Waiting 21s before retry 1/3...
Rate limited. Waiting 60s before retry 2/3...
Rate limited. Waiting 60s before retry 2/3...


### Save Data to Databases

In [ ]:
'''
-- Create tables for hard and soft skills with embeddings

CREATE TABLE soft_skills (
    id          SERIAL PRIMARY KEY,
    job_id      VARCHAR(255) NOT NULL REFERENCES job_postings(id) ON DELETE CASCADE,
    skill_description TEXT NOT NULL,
    weight      FLOAT,
    embedding   VECTOR(3072)  -- gemini-embedding-001 output dimension
);

CREATE TABLE hard_skills (
    id              SERIAL PRIMARY KEY,
    job_id          VARCHAR(255) NOT NULL REFERENCES job_postings(id) ON DELETE CASCADE,
    skill_description TEXT NOT NULL,
    time_experience FLOAT,
    weight          FLOAT,
    embedding       VECTOR(3072)
);
'''

In [ ]:
from pgvector.psycopg2 import register_vector
import psycopg2

database_user = getenv("DB_USER")
database_password = getenv("DB_PASSWORD")
HOST = "localhost"
DATABASE = "market_fit"

def save_hard_skills(hard_skills_df: pd.DataFrame):
    with psycopg2.connect(
        host=HOST,
        database=DATABASE,
        user=database_user,
        password=database_password
    ) as conn:
        if conn:
            print("Connection to the database was successful!")
            conn.autocommit = True
            register_vector(conn)

        cur = conn.cursor()
        for _, row in hard_skills_df.iterrows():
            insert_query = """
                INSERT INTO hard_skills (job_id, skill_description, time_experience, weight, embedding)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            values = (
                row["job_id"],
                row["skill_description"],
                row.get("time_experience"),
                row.get("weight"),
                row["embedding"]
            )
            cur.execute(insert_query, values)

def save_soft_skills(soft_skills_df: pd.DataFrame):
    with psycopg2.connect(
        host=HOST,
        database=DATABASE,
        user=database_user,
        password=database_password
    ) as conn:
        if conn:
            print("Connection to the database was successful!")
            conn.autocommit = True
            register_vector(conn)

        cur = conn.cursor()
        for _, row in soft_skills_df.iterrows():
            insert_query = """
                INSERT INTO soft_skills (job_id, skill_description, weight, embedding)
                VALUES (%s, %s, %s, %s)
                ON CONFLICT DO NOTHING
            """
            values = (
                row["job_id"],
                row["skill_description"],
                row.get("weight"),
                row["embedding"]
            )
            cur.execute(insert_query, values)

# ── Run ───────────────────────────────────────────────────────────────────────

#save_hard_skills(hard_skills_df_with_embeddings)
#save_soft_skills(soft_skills_df_with_embeddings)

Connection to the database was successful!
Connection to the database was successful!
